In [ ]:
from core.file_manager import preprocess_file_manager
from core.visualization_lib import folder_shower, normalize_volume
from core.helper import  copy_NiFty, patients_transform

from core.transformers.nifti_to_raw_transformer import nifti_to_raw_transformer
from core.transformers.masks_fill_transformer import masks_fill_transformer
from core.transformers.crop_transformer import non_weighted_crop_transformer
from core.transformers.resample_transformer import resample_transformer
import  core.transformers.normalizers as normalizers
from core.transformers.normalizers.z_score_normalizer_transformer import z_score_normalizer_transformer
from core.transformers.normalizers.min_max_normalizer_transformer import min_max_normalizer_transformer
from settings.main_settings import test_settings


In [ ]:
settings = test_settings().get_setting_dictionary()
preprocessed_steps = settings['preprocessed_steps']
preprocessing_steps_list = settings['preprocessing_steps_list']
channels = settings['channels']
original_data_folder = settings['original_data_folder']
target_spacing = settings['target_spacing']
crop_size = settings['crop_size']
# filter=['3322','001','003']
filter = None

file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [ ]:
step_functions = {}

In [ ]:

step_functions['start'] = lambda: copy_NiFty(
    original_data_folder,
    file_manager,
    channels,
    filter=filter,
    step=preprocessed_steps[preprocessing_steps_list[1][0]]['start']
)

In [ ]:
if preprocessed_steps.get('min_max_normalization'):
    start_step = preprocessed_steps['min_max_normalization']['start']
    end_step = preprocessed_steps['min_max_normalization']['end']
    min_max_transformer = min_max_normalizer_transformer(channels_to_normalize=settings['min_max_channels_to_normalize'],mask_channel=settings['mask_channel'])
    step_functions['min_max_normalization'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, min_max_transformer,filter=filter)

In [ ]:

if preprocessed_steps.get('z_score_normalization'):
    start_step = preprocessed_steps['z_score_normalization']['start']
    end_step = preprocessed_steps['z_score_normalization']['end']
    normalization_transformer = z_score_normalizer_transformer(channels_to_normalize=settings['z_score_channels_to_normalize'],mask_channel=settings['mask_channel'])

    step_functions['z_score_normalization'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, normalization_transformer,filter=filter)

In [ ]:

if preprocessed_steps.get('resampling'):
    start_step = preprocessed_steps['resampling']['start']
    end_step = preprocessed_steps['resampling']['end']
    resample_transformer = resample_transformer(target_spacing=target_spacing,crop_size=crop_size,channels=channels)

    step_functions['resampling'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, resample_transformer,filter=filter)

In [ ]:
if preprocessed_steps.get('nifti_to_raw'):
    start_step = preprocessed_steps['nifti_to_raw']['start']
    end_step = preprocessed_steps['nifti_to_raw']['end']
    to_raw_transform = nifti_to_raw_transformer()

    step_functions['nifti_to_raw'] =  lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, to_raw_transform,filter=filter)

In [ ]:
if preprocessed_steps.get('filling_anatomy_gaps'):
    start_step = preprocessed_steps['filling_anatomy_gaps']['start']
    end_step = preprocessed_steps['filling_anatomy_gaps']['end']
    gap_fill_transformer = masks_fill_transformer()
    
    step_functions['filling_anatomy_gaps'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, gap_fill_transformer,filter=filter)

In [ ]:
if preprocessed_steps.get('cropping'):
    start_step = preprocessed_steps['cropping']['start']
    end_step = preprocessed_steps['cropping']['end']
    nw_crop_transformer = non_weighted_crop_transformer(crop_size)

    step_functions['cropping'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, nw_crop_transformer,filter=filter)

In [ ]:
for step, _ in preprocessing_steps_list:
    print(f"Starting step: {step}")
    file_manager.current_load_step = preprocessed_steps[step]['start']
    print(len(file_manager.get_file_names()))
    # step_functions[step]()
    print(f"Finished step: {step}")